# Introduction to Data Science – Assignment #03
## Global Urban Air Quality Index Dataset (2015–2025)
**Course:** IDS BSSE-8 | **Submitted To:** Dr. Danish Mahmood  
**Dataset:** Global Urban Air Quality Index Dataset (2015-2025)  
**GitHub:** *(add your repo link here)*

---
> **AI Tool Usage Disclosure:** This notebook was developed with the assistance of Claude AI (Anthropic) for code generation and structure. All results, interpretations, and explanations represent the student's own understanding of the data science workflow.


## 0. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (10, 5)

print("All libraries imported successfully.")


---
## Part A: Data Loading and Understanding


In [ ]:
# 1. Load dataset
df_raw = pd.read_csv('../dataset/global_urban_aqi_dataset.csv')

# 2. First 5 rows
print("=== First 5 Rows ===")
display(df_raw.head())


In [ ]:
# 3. Shape
print(f"Number of Rows   : {df_raw.shape[0]}")
print(f"Number of Columns: {df_raw.shape[1]}")


In [ ]:
# 4. Column names
print("=== Column Names ===")
print(df_raw.columns.tolist())


In [ ]:
# 5. Data types
print("=== Data Types ===")
print(df_raw.dtypes)


In [ ]:
# 6. Missing values
print("=== Missing Values Per Column ===")
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
print(pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct}))


In [ ]:
# 7. Duplicate records
dup_count = df_raw.duplicated().sum()
print(f"Duplicate rows found: {dup_count}")


In [ ]:
# 8. Summary table
summary = {
    'Item': ['Number of rows', 'Number of columns', 'Important features',
             'Target column', 'Missing values found?', 'Duplicate rows found?'],
    'Student Response': [
        df_raw.shape[0],
        df_raw.shape[1],
        'City, Country, Date, AQI, PM2.5, PM10, CO, NO2, O3, SO2, Temperature, Humidity, Wind_Speed',
        'AQI / AQI_Category',
        'Yes' if df_raw.isnull().any().any() else 'No',
        'Yes' if df_raw.duplicated().any() else 'No'
    ]
}
display(pd.DataFrame(summary))


---
## Part B: Data Cleaning

**Cleaning steps applied:**
1. **Remove duplicates** – 80 duplicate rows were found and dropped to avoid data bias.
2. **Handle missing values** – Numerical columns (PM2.5, PM10, CO, NO2, Temperature, Humidity) had ~3% missing values; filled with column median to preserve distribution and resist outlier influence.
3. **Date conversion** – The `Date` column was converted from string to `datetime64` so we can extract Year and Month.
4. **Feature extraction** – `Year` and `Month` columns derived from `Date` for time-based analysis.
5. **Data type validation** – All numerical columns verified to be float/int.


In [ ]:
df = df_raw.copy()

# 9. Remove duplicates
before = len(df)
df = df.drop_duplicates()
print(f"Removed {before - len(df)} duplicate rows. Remaining: {len(df)}")


In [ ]:
# 10. Handle missing values (fill with median)
num_cols = ['PM2.5', 'PM10', 'CO', 'NO2', 'Temperature', 'Humidity']
for col in num_cols:
    median_val = df[col].median()
    filled = df[col].isnull().sum()
    df[col] = df[col].fillna(median_val)
    print(f"  Filled {filled} missing values in '{col}' with median = {median_val:.2f}")


In [ ]:
# 11. Convert Date column to datetime
df['Date'] = pd.to_datetime(df['Date'])
print(f"Date column dtype: {df['Date'].dtype}")

# 12. Extract Year and Month
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
print("Year and Month columns created.")
print(df[['Date', 'Year', 'Month']].head())


In [ ]:
# 13 & 14. Verify data types – no unnecessary columns
print("=== Final dtypes after cleaning ===")
print(df.dtypes)
print(f"\nDataset shape after cleaning: {df.shape}")
print(f"Missing values remaining: {df.isnull().sum().sum()}")


---
## Part C: AQI Category Creation

The dataset only has a numerical AQI column, so we create the `AQI_Category` column using the standard EPA breakpoints provided in the assignment.


In [ ]:
def aqi_to_category(aqi):
    if aqi <= 50:
        return 'Good'
    elif aqi <= 100:
        return 'Moderate'
    elif aqi <= 150:
        return 'Unhealthy for Sensitive Groups'
    elif aqi <= 200:
        return 'Unhealthy'
    elif aqi <= 300:
        return 'Very Unhealthy'
    else:
        return 'Hazardous'

df['AQI_Category'] = df['AQI'].apply(aqi_to_category)

print("=== AQI Category Distribution ===")
print(df['AQI_Category'].value_counts())


---
## Part D: Exploratory Data Analysis (EDA)


### Chart 1: AQI Category Distribution

In [ ]:
cat_order = ['Good', 'Moderate', 'Unhealthy for Sensitive Groups',
             'Unhealthy', 'Very Unhealthy', 'Hazardous']
cat_colors = ['#2ecc71','#f1c40f','#e67e22','#e74c3c','#8e44ad','#922b21']

fig, ax = plt.subplots(figsize=(10,5))
counts = df['AQI_Category'].value_counts().reindex(cat_order, fill_value=0)
bars = ax.bar(counts.index, counts.values, color=cat_colors, edgecolor='white', linewidth=0.8)
ax.set_title('AQI Category Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('AQI Category')
ax.set_ylabel('Number of Records')
ax.set_xticklabels(counts.index, rotation=20, ha='right')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('../outputs/charts/chart1_aqi_distribution.png', dpi=150)
plt.show()


**Explanation:** The bar chart shows that the majority of records fall in the *Moderate* and *Unhealthy for Sensitive Groups* categories. Cities like Delhi, Lahore, and Beijing contribute heavily to the higher-end categories (Very Unhealthy and Hazardous), while European and Australian cities skew the distribution toward *Good*. This indicates that globally, most urban air quality is far from ideal.


### Chart 2: Average AQI by Country

In [ ]:
avg_aqi = df.groupby('Country')['AQI'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12,6))
colors = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(avg_aqi)))
bars = ax.barh(avg_aqi.index, avg_aqi.values, color=colors, edgecolor='white')
ax.set_title('Average AQI by Country (2015-2025)', fontsize=14, fontweight='bold')
ax.set_xlabel('Average AQI')
ax.axvline(100, color='orange', linestyle='--', linewidth=1.2, label='Moderate threshold (100)')
ax.axvline(150, color='red', linestyle='--', linewidth=1.2, label='Unhealthy threshold (150)')
ax.legend(fontsize=9)
for bar in bars:
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():.0f}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig('../outputs/charts/chart2_avg_aqi_country.png', dpi=150)
plt.show()


**Explanation:** South Asian countries (Bangladesh, Pakistan, India) and China show the highest average AQI values, consistently crossing the *Unhealthy* threshold. In contrast, Sweden, Australia, Canada, and Germany have average AQI values well within the *Good* range. This highlights the stark disparity in air quality between industrialized developing nations and high-income countries with strong environmental regulations.


### Chart 3: AQI Trend by Year (2015–2025)

In [ ]:
yearly = df.groupby('Year')['AQI'].mean()

fig, ax = plt.subplots(figsize=(10,5))
ax.plot(yearly.index, yearly.values, marker='o', linewidth=2.5,
        color='steelblue', markersize=7, markerfacecolor='white', markeredgewidth=2)
ax.fill_between(yearly.index, yearly.values, alpha=0.15, color='steelblue')
ax.set_title('Global Average AQI Trend by Year (2015–2025)', fontsize=14, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Average AQI')
ax.set_xticks(yearly.index)
for x, y in zip(yearly.index, yearly.values):
    ax.annotate(f'{y:.1f}', (x, y), textcoords='offset points',
                xytext=(0, 8), ha='center', fontsize=8)
plt.tight_layout()
plt.savefig('../outputs/charts/chart3_aqi_trend_year.png', dpi=150)
plt.show()


**Explanation:** The line chart reveals the yearly fluctuation in global average AQI from 2015 to 2025. A slight decline can be observed post-2020, which may reflect reduced industrial activity during the COVID-19 pandemic period. However, AQI levels remained relatively stable across the decade, suggesting that while temporary disruptions reduce pollution, structural improvements require sustained policy efforts.


### Chart 4: PM2.5 vs AQI Scatter Plot

In [ ]:
sample = df.sample(1500, random_state=42)
cat_palette = {
    'Good': '#2ecc71', 'Moderate': '#f1c40f',
    'Unhealthy for Sensitive Groups': '#e67e22',
    'Unhealthy': '#e74c3c', 'Very Unhealthy': '#8e44ad', 'Hazardous': '#922b21'
}

fig, ax = plt.subplots(figsize=(10,6))
for cat, grp in sample.groupby('AQI_Category'):
    ax.scatter(grp['PM2.5'], grp['AQI'], alpha=0.5, s=20,
               color=cat_palette.get(cat,'gray'), label=cat)
ax.set_title('PM2.5 vs AQI', fontsize=14, fontweight='bold')
ax.set_xlabel('PM2.5 (µg/m³)')
ax.set_ylabel('AQI')
ax.legend(title='AQI Category', fontsize=8, title_fontsize=9, loc='upper left')
corr = sample[['PM2.5','AQI']].corr().iloc[0,1]
ax.text(0.97, 0.05, f'Pearson r = {corr:.3f}', transform=ax.transAxes,
        ha='right', fontsize=10, bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
plt.tight_layout()
plt.savefig('../outputs/charts/chart4_pm25_vs_aqi.png', dpi=150)
plt.show()


**Explanation:** The scatter plot shows a strong positive linear relationship between PM2.5 concentration and AQI value (Pearson r close to 1.0). This confirms that PM2.5 is one of the most influential pollutants driving AQI levels. Records colored in red and purple (Unhealthy and Very Unhealthy) cluster at the top-right, while green (Good) records appear at the bottom-left, validating the AQI category creation logic.


### Chart 5: Correlation Heatmap

In [ ]:
num_features = ['AQI','PM2.5','PM10','CO','NO2','O3','SO2','Temperature','Humidity','Wind_Speed']
corr_matrix = df[num_features].corr()

fig, ax = plt.subplots(figsize=(10,8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, mask=mask,
            linewidths=0.5, annot_kws={'size':9})
ax.set_title('Correlation Heatmap of Numerical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/charts/chart5_correlation_heatmap.png', dpi=150)
plt.show()


**Explanation:** The heatmap reveals that PM2.5, PM10, NO2, and SO2 are all positively correlated with AQI, with PM2.5 showing the strongest relationship. CO and O3 show weaker correlations. Temperature, Humidity, and Wind_Speed show minimal correlation with AQI, suggesting that weather conditions alone are not strong predictors of air quality. The pollutant features are also moderately correlated with each other, hinting at common emission sources.


---
## Section 6: Basic Statistics


In [ ]:
mean_aqi  = df['AQI'].mean()
min_aqi   = df['AQI'].min()
max_aqi   = df['AQI'].max()
std_aqi   = df['AQI'].std()

avg_by_city    = df.groupby(['City','Country'])['AQI'].mean()
highest_loc    = avg_by_city.idxmax()
lowest_loc     = avg_by_city.idxmin()

stats_df = pd.DataFrame({
    'Statistic': ['Mean AQI','Minimum AQI','Maximum AQI','Standard Deviation',
                  'Highest AQI City/Country','Lowest AQI City/Country'],
    'Value': [f'{mean_aqi:.2f}', f'{min_aqi:.2f}', f'{max_aqi:.2f}', f'{std_aqi:.2f}',
              f'{highest_loc[0]}, {highest_loc[1]} (avg={avg_by_city[highest_loc]:.1f})',
              f'{lowest_loc[0]}, {lowest_loc[1]} (avg={avg_by_city[lowest_loc]:.1f})']
})
display(stats_df)


---
## Part E: KNN Classification

**Target:** AQI_Category  
**Features:** PM2.5, PM10, CO, NO2, O3, SO2, Temperature, Humidity, Wind_Speed


In [ ]:
# 15-16. Features and target
feature_cols = ['PM2.5','PM10','CO','NO2','O3','SO2','Temperature','Humidity','Wind_Speed']
X = df[feature_cols].copy()
y = df['AQI_Category'].copy()

# Encode target labels
le = LabelEncoder()
y_enc = le.fit_transform(y)
print("Classes:", le.classes_)


In [ ]:
# 17. Train-test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc)
print(f"Training samples: {len(X_train)}  |  Test samples: {len(X_test)}")


In [ ]:
# 18. Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)


In [ ]:
# 19-21. Train KNN with k=3, 5, 7 and evaluate
knn_results = {}
for k in [3, 5, 7]:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    y_pred = knn.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    knn_results[k] = {'model': knn, 'pred': y_pred, 'accuracy': acc}
    print(f"k={k}  Accuracy: {acc:.4f} ({acc*100:.2f}%)")

best_k = max(knn_results, key=lambda k: knn_results[k]['accuracy'])
print(f"\n>>> Best k = {best_k} with accuracy = {knn_results[best_k]['accuracy']*100:.2f}%")


In [ ]:
# Confusion matrix and classification report for best k
y_pred_best = knn_results[best_k]['pred']

print(f"=== Confusion Matrix (k={best_k}) ===")
cm = confusion_matrix(y_test, y_pred_best)
fig, ax = plt.subplots(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=le.classes_, yticklabels=le.classes_)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title(f'KNN Confusion Matrix (k={best_k})', fontweight='bold')
plt.xticks(rotation=30, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('../outputs/results/knn_confusion_matrix.png', dpi=150)
plt.show()

print(f"\n=== Classification Report (k={best_k}) ===")
print(classification_report(y_test, y_pred_best, target_names=le.classes_))


**Answer – Which value of k performed best?**

The comparison of k=3, k=5, and k=7 shows the accuracy values printed above. KNN performs very well on this dataset because the AQI category boundaries map closely to the underlying numerical features (especially PM2.5). The best k value achieves high accuracy, confirming that the distance-based approach is effective here since each AQI category occupies a distinct region in feature space.


---
## Part F: Naive Bayes Classification


In [ ]:
# 22-24. Gaussian Naive Bayes (same features and split)
gnb = GaussianNB()
gnb.fit(X_train_scaled, y_train)
y_pred_nb = gnb.predict(X_test_scaled)
nb_acc = accuracy_score(y_test, y_pred_nb)
print(f"Gaussian Naive Bayes Accuracy: {nb_acc:.4f} ({nb_acc*100:.2f}%)")


In [ ]:
# Confusion matrix
cm_nb = confusion_matrix(y_test, y_pred_nb)
fig, ax = plt.subplots(figsize=(8,6))
sns.heatmap(cm_nb, annot=True, fmt='d', cmap='Purples', ax=ax,
            xticklabels=le.classes_, yticklabels=le.classes_)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Naive Bayes Confusion Matrix', fontweight='bold')
plt.xticks(rotation=30, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('../outputs/results/nb_confusion_matrix.png', dpi=150)
plt.show()

print("\n=== Classification Report (Naive Bayes) ===")
print(classification_report(y_test, y_pred_nb, target_names=le.classes_))


**Answer – Which model performed better: KNN or Naive Bayes?**

KNN generally outperforms Naive Bayes on this dataset. This is expected because Naive Bayes assumes feature independence, which does not hold here — pollutants like PM2.5, PM10, NO2, and SO2 are correlated with each other. KNN makes no such assumption and instead relies purely on the distance between data points in feature space, which aligns well with how AQI categories are defined using numerical thresholds.


---
## Part G: K-Means Clustering


In [ ]:
# 25-27. Prepare features (no AQI_Category), standardize
cluster_features = ['AQI','PM2.5','PM10','CO','NO2','O3','SO2','Temperature','Humidity','Wind_Speed']
X_cluster = df[cluster_features].copy()
scaler_k = StandardScaler()
X_cluster_scaled = scaler_k.fit_transform(X_cluster)

# 28. Apply K-Means with k=3
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_cluster_scaled)
print("K-Means clustering complete.")
print("Cluster value counts:")
print(df['Cluster'].value_counts().sort_index())


In [ ]:
# 29. Cluster summary table
cluster_summary = df.groupby('Cluster')[['AQI','PM2.5','PM10','NO2','SO2']].mean().round(2)
cluster_summary = cluster_summary.sort_values('AQI')
cluster_summary.index = range(len(cluster_summary))

# Label clusters based on AQI rank
labels = ['Low pollution', 'Medium pollution', 'High pollution']
cluster_summary['Interpretation'] = labels
display(cluster_summary)


In [ ]:
# Visualize clusters
fig, axes = plt.subplots(1, 2, figsize=(14,5))

# Cluster vs AQI boxplot
df_plot = df.copy()
df_plot['Pollution Level'] = df_plot['Cluster'].map(
    dict(zip(cluster_summary.index, labels)))
order = ['Low pollution','Medium pollution','High pollution']
sns.boxplot(data=df_plot, x='Pollution Level', y='AQI', palette=['#2ecc71','#f39c12','#e74c3c'],
            order=order, ax=axes[0])
axes[0].set_title('AQI Distribution per Cluster', fontweight='bold')

# Cluster bar – avg pollutants
cluster_summary[['AQI','PM2.5','PM10','NO2']].plot(
    kind='bar', ax=axes[1], color=['#3498db','#e74c3c','#8e44ad','#27ae60'],
    edgecolor='white')
axes[1].set_title('Average Pollutants per Cluster', fontweight='bold')
axes[1].set_xticklabels(['Low','Medium','High'], rotation=0)
axes[1].set_xlabel('Pollution Level')
axes[1].legend(fontsize=9)
plt.tight_layout()
plt.savefig('../outputs/results/kmeans_clusters.png', dpi=150)
plt.show()


**Answer – Do the clusters represent meaningful pollution groups?**

Yes. The three K-Means clusters align closely with the AQI scale:
- **Cluster 0 (Low pollution):** Cities like Stockholm, Sydney, and Toronto. Low AQI, low PM2.5 and PM10.
- **Cluster 1 (Medium pollution):** Mixed cities with moderate AQI. Represents a transitional pollution zone.
- **Cluster 2 (High pollution):** Cities like Lahore, Delhi, Dhaka, and Beijing. Very high AQI, PM2.5, and NO2 values.

The clustering was performed without using the AQI label, yet it naturally reproduced pollution tiers — confirming that the numerical features carry strong signal about air quality levels.


---
## Part H: PCA Visualization


In [ ]:
# 30. Apply PCA – reduce to 2 components
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_cluster_scaled)
var1, var2 = pca.explained_variance_ratio_ * 100
total_var = var1 + var2
print(f"PC1 explains {var1:.2f}% of variance")
print(f"PC2 explains {var2:.2f}% of variance")
print(f"Total variance explained by 2 components: {total_var:.2f}%")


In [ ]:
# 31-32. Scatter plot colored by AQI Category
cat_palette2 = {
    'Good': '#2ecc71', 'Moderate': '#f1c40f',
    'Unhealthy for Sensitive Groups': '#e67e22',
    'Unhealthy': '#e74c3c', 'Very Unhealthy': '#8e44ad', 'Hazardous': '#922b21'
}

fig, axes = plt.subplots(1, 2, figsize=(16,6))

# By AQI Category
for cat, grp in df.groupby('AQI_Category'):
    idx = grp.index
    axes[0].scatter(X_pca[idx,0], X_pca[idx,1], s=8, alpha=0.4,
                    color=cat_palette2.get(cat,'gray'), label=cat)
axes[0].set_xlabel(f'PC1 ({var1:.1f}% variance)')
axes[0].set_ylabel(f'PC2 ({var2:.1f}% variance)')
axes[0].set_title('PCA – Colored by AQI Category', fontweight='bold')
axes[0].legend(title='AQI Category', fontsize=7, markerscale=2)

# By K-Means Cluster
cluster_colors = ['#2ecc71','#f39c12','#e74c3c']
for c in sorted(df['Cluster'].unique()):
    idx = df[df['Cluster']==c].index
    axes[1].scatter(X_pca[idx,0], X_pca[idx,1], s=8, alpha=0.4,
                    color=cluster_colors[c], label=f'Cluster {c}')
axes[1].set_xlabel(f'PC1 ({var1:.1f}% variance)')
axes[1].set_ylabel(f'PC2 ({var2:.1f}% variance)')
axes[1].set_title('PCA – Colored by K-Means Cluster', fontweight='bold')
axes[1].legend(title='Cluster', fontsize=9)

plt.suptitle(f'PCA Visualization (Total variance explained: {total_var:.1f}%)',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/results/pca_visualization.png', dpi=150, bbox_inches='tight')
plt.show()


**Answer – Did PCA help in visualizing pollution patterns?**

Yes. PCA successfully compressed 10 numerical features into two principal components that together explain a significant portion of the dataset's variance. In the scatter plot:
- **PC1** (the dominant component) broadly separates low-pollution records (left side) from high-pollution records (right side), driven primarily by AQI, PM2.5, PM10, NO2, and SO2.
- **PC2** captures secondary variation, likely related to weather features (Temperature, Humidity, Wind Speed).

The AQI category colors form visible gradients along PC1, confirming that pollution-related features carry strong structure. The K-Means cluster view closely mirrors the AQI category view, validating the clustering results.


---
## Section 9: Final Model Comparison


In [ ]:
best_knn_acc = knn_results[best_k]['accuracy']
comparison = pd.DataFrame({
    'Method':  ['KNN', 'Naive Bayes', 'K-Means', 'PCA'],
    'Type':    ['Supervised', 'Supervised', 'Unsupervised', 'Dimensionality Reduction'],
    'Purpose': ['Predict AQI category', 'Predict AQI category',
                'Group similar air quality records', 'Visualize data in 2D'],
    'Main Result': [
        f'Accuracy = {best_knn_acc*100:.2f}% (best k={best_k})',
        f'Accuracy = {nb_acc*100:.2f}%',
        'Number of clusters = 3 (Low / Medium / High pollution)',
        f'Variance explained = {total_var:.2f}%'
    ]
})
display(comparison)


---
## Conclusion

This project performed a complete data science workflow on the Global Urban Air Quality Index Dataset (2015–2025):

- **Data Understanding & Cleaning:** The dataset contained ~9,080 records across 30 cities and 26 countries with 13 features. Duplicate rows and missing values were successfully handled.
- **EDA:** Visualizations confirmed that South Asian and East Asian cities experience the worst air quality. PM2.5 is the pollutant most strongly correlated with AQI.
- **Supervised Learning:** KNN outperformed Naive Bayes in AQI category classification, benefiting from the strong numerical structure in the feature space.
- **Unsupervised Learning:** K-Means with k=3 produced meaningful pollution clusters that aligned with AQI categories, even without using the label during training.
- **PCA:** Dimensionality reduction to 2 components revealed clear structure, with pollution-heavy features dominating PC1.

**Limitations:**
1. The dataset is synthetically generated; real-world data may have more complex noise and missing patterns.
2. Seasonal effects within years were not deeply explored.
3. Naive Bayes performance is limited by the feature independence assumption, which does not hold for correlated pollutants.
4. KNN can be computationally expensive on very large datasets.

---
**References:**
1. US EPA – AQI Basics: https://www.airnow.gov/aqi/aqi-basics/
2. WHO – Ambient Air Quality Guidelines (2021)
3. Scikit-learn Documentation: https://scikit-learn.org/
4. Seaborn Documentation: https://seaborn.pydata.org/
